In [2]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, GridSearchCV, learning_curve
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, classification_report


In [3]:
train = pd.read_csv('../6.Data/Yann_Process_train.csv')
test = pd.read_csv('../6.Data/Yann_Process_test.csv')

selected_features = [
    "chargebacks_12m",
    "failed_payments_6m",
    "device_trust_z",
    "ip_risk_z",
    "max_to_avg_ratio",
    "is_new_device",
    "is_vpn"
]

# Garde uniquement ces colonnes dans train
train = train[selected_features + ["target_is_fraud"]]

# Garde uniquement ces colonnes dans test
test = test[selected_features]

print("Train shape:", train.shape)
print("Test shape:", test.shape)


KeyError: "['is_new_device', 'is_vpn'] not in index"

In [ ]:


target = 'target_is_fraud'
id_col = 'customer_id'

X = train.drop(columns=[target, id_col])
y = train[target]
X_test = test.drop(columns=[id_col])

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

In [ ]:
rf = RandomForestClassifier(random_state=42)

param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [1, 10]
}

grid = GridSearchCV(rf, param_grid, cv=5, scoring='f1', n_jobs=-1)
grid.fit(X_train, y_train)

best_model = grid.best_estimator_
print('Best params:', grid.best_params_)


In [ ]:
y_pred = best_model.predict(X_val)

print('Confusion Matrix:')
print(confusion_matrix(y_val, y_pred))

print('\nClassification Report:')
print(classification_report(y_val, y_pred))


In [ ]:
train_sizes, train_scores, val_scores = learning_curve(
    best_model, X, y, cv=3, scoring='f1', n_jobs=-1
)

plt.plot(train_sizes, train_scores.mean(axis=1))
plt.plot(train_sizes, val_scores.mean(axis=1))
plt.xlabel('Training Size')
plt.ylabel('F1 Score')
plt.show()

In [ ]:
# Prédiction binaire 0/1 pour Kaggle
test_predictions = best_model.predict(X_test)

submission = pd.DataFrame({
    "customer_id": test_df["customer_id"],
    "target_is_fraud": test_predictions
})

submission.to_csv('../submissions/submission_rf_simple.csv', index=False)
print('Submission exportée.')


In [ ]:
test_df = pd.read_csv('../data/test.csv')